# Malinois MPRA 3: a real sequence-only baseline (deltaSVM), and P(bound) on top of it

motifbreakR barely ranks the Malinois allelic effects (AUROC for clear effects 0.49–0.55), so beating it says little.
Here the baseline is **deltaSVM** (Lee et al. 2015, Nat Genet): a gapped k-mer SVM (gkm-SVM, Ghandi et al. 2014,
software LS-GKM) trained to tell a cell's DNase-seq open regions from GC-matched genomic background, and a variant's
score = sum over the 10 ten-mers covering it of w(alt 10-mer) − w(ref 10-mer). It only reads DNA sequence, is not
deep learning, and is cell-specific (one model per cell).

**Neither deltaSVM nor the CAV sees any MPRA label.** deltaSVM is trained on ENCODE DNase-seq peaks (GRCh38):
HepG2 `ENCFF897NME`, K562 `ENCFF185XRG`, SK-N-SH `ENCFF752OZB`; top 20,000 peaks by signal, 300 bp around the peak
centre; negatives = random 300 bp genomic windows outside that cell's peaks, matched on GC (bins of 0.02).
Options `-t 2 -l 10 -k 6 -d 3` (the deltaSVM setting). Chromosome 2 is held out of training only to print a
sanity AUC.

**Section 1 — train one cell (CPU).** Set `CELL` in 1.1, run Section 1. Three CPU sessions (one per cell) can run
at the same time. Roughly 1–3 h per cell, mostly `gkmtrain`; the model, the 10-mer weights and the variant scores
are saved to `malinois/deltasvm/`, and a finished step is skipped when re-run.

**Section 2 — evaluation (CPU, after all three cells).** For each cell (HepG2, K562, SK-N-SH) and variant set
(all / active):

| | question |
|---|---|
| 2.2 baseline | how well deltaSVM, motifbreakR and summed motifbreakR predict Δ (Pearson, Spearman, AUROC for clear effects); which cell's deltaSVM predicts which cell's Δ |
| 2.3 P on top | out-of-sample Pearson gain (chromosome halves) of deltaSVM + P (P gating deltaSVM) over deltaSVM; of deltaSVM + motifbreakR + P over deltaSVM + motifbreakR; within-TF b(deltaSVM × P); AUROC gain for clear effects with a 95% CI |
| 2.3 contrast | HepG2 or K562 minus SK-N-SH of the within-TF b(deltaSVM × P), 95% CI |

As in `Malinois MPRA 2`, the TF tiers by held-out ChIP AUC (all, ≥ 0.70, ≥ 0.75, ≥ 0.80) are fixed in advance, P
is used as its within-TF percentile, and the null N2 (P shuffled among the calls of the same TF) is run through the
whole computation.

In [ ]:
![ -x /content/lsgkm/src/gkmtrain ] || (git clone -q --depth 1 https://github.com/Dongwon-Lee/lsgkm /content/lsgkm && make -s -C /content/lsgkm/src > /dev/null 2>&1)
!pip install -q pyfaidx
from google.colab import drive
drive.mount('/content/drive')

---
# Section 1 — train deltaSVM for one cell *(CPU runtime)*

## 1.1 Settings and the DNase peaks

In [ ]:
CELL = "HepG2"          # "HepG2", "K562" or "SKNSH": one CPU session per cell, all three can run at once

WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, time, shutil, subprocess
import numpy as np, pandas as pd
from pyfaidx import Fasta
from sklearn.metrics import roc_auc_score

C = json.load(open(f"{WORK_DIR}/config.json"))
MAL = f"{WORK_DIR}/malinois"
DS = f"{MAL}/deltasvm"
os.makedirs(DS, exist_ok=True)
DNASE = {"HepG2": "ENCFF897NME", "K562": "ENCFF185XRG", "SKNSH": "ENCFF752OZB"}   # ENCODE DNase-seq peaks, GRCh38
N_POS, WIDTH, TEST_CHROM, SEED, L = 20000, 300, "chr2", 0, 10
GKM = "/content/lsgkm/src"
GKM_OPTS = "-t 2 -l 10 -k 6 -d 3 -m 4000 -T 4"
CHROMS = [f"chr{i}" for i in range(1, 23)] + ["chrX"]
TAG = f"{DS}/{CELL}"
LOC = f"/content/{CELL}"
assert os.path.exists(f"{GKM}/gkmtrain"), "lsgkm did not compile; rerun the first cell"

LOCAL_FA = "/content/hg38.fa"
if not os.path.exists(LOCAL_FA):
    print("copying hg38 to local disk (~3 GB, one-off)...")
    shutil.copy(C["genome_fa"], LOCAL_FA)
    if os.path.exists(C["genome_fa"] + ".fai"):
        shutil.copy(C["genome_fa"] + ".fai", LOCAL_FA + ".fai")
GENOME = Fasta(LOCAL_FA, sequence_always_upper=True)

BED = f"{DS}/{DNASE[CELL]}.bed.gz"
if not os.path.exists(BED):
    url = f"https://www.encodeproject.org/files/{DNASE[CELL]}/@@download/{DNASE[CELL]}.bed.gz"
    subprocess.run(["wget", "-q", "-O", BED + ".part", url], check=True)
    os.replace(BED + ".part", BED)
PK = pd.read_csv(BED, sep="\t", header=None, usecols=[0, 1, 2, 6], names=["chrom", "start", "end", "signal"])
PK = PK[PK["chrom"].isin(CHROMS)].reset_index(drop=True)
PK["mid"] = (PK["start"] + PK["end"]) // 2
print(f"{CELL}: {len(PK):,} DNase peaks in {DNASE[CELL]}; median width {int((PK['end'] - PK['start']).median())} bp")

## 1.2 Positives (top peaks) and GC-matched negatives

In [ ]:
rng = np.random.default_rng(SEED)
want = PK.sort_values("signal", ascending=False).head(int(N_POS * 1.1))
n_cand = 15 * N_POS
tot = sum(len(GENOME[ch]) for ch in CHROMS)
POS, NEG = [], []
for ch in CHROMS:
    s = GENOME[ch][:].seq
    cov = np.zeros(len(s) // 100 + 2, bool)                     # 100 bp bins touched by any peak of this cell
    pk = PK[PK["chrom"] == ch]
    for a, b in zip(pk["start"].values, pk["end"].values):
        cov[a // 100:b // 100 + 1] = True
    w = want[want["chrom"] == ch]
    for mid, sig in zip(w["mid"].values, w["signal"].values):
        a = int(mid) - WIDTH // 2
        x = s[a:a + WIDTH]
        if a >= 0 and len(x) == WIDTH and "N" not in x:
            POS.append((ch, a, x, sig))
    for a in rng.integers(0, len(s) - WIDTH, int(n_cand * len(s) / tot)):
        if not cov[a // 100:(a + WIDTH) // 100 + 1].any():
            x = s[a:a + WIDTH]
            if "N" not in x:
                NEG.append((ch, int(a), x))
    del s
POS = pd.DataFrame(POS, columns=["chrom", "start", "seq", "signal"]).sort_values("signal", ascending=False).head(N_POS)
NEG = pd.DataFrame(NEG, columns=["chrom", "start", "seq"])
gc = lambda ss: np.array([(x.count("G") + x.count("C")) / WIDTH for x in ss])
POS["gc"], NEG["gc"] = gc(POS["seq"]), gc(NEG["seq"])
BINS = np.linspace(0, 1, 51)


def match(p_, n_):
    hp, hn = np.digitize(p_["gc"], BINS), np.digitize(n_["gc"], BINS)
    take = []
    for b, cnt in zip(*np.unique(hp, return_counts=True)):
        j = np.flatnonzero(hn == b)
        take.extend(rng.choice(j, min(cnt, len(j)), replace=False))
    return n_.iloc[np.sort(take)]


def write_fa(df, path):
    with open(path, "w") as f:
        for r in df.itertuples():
            f.write(f">{r.chrom}:{r.start}\n{r.seq}\n")


rows = []
for split, t in (("train", False), ("test (" + TEST_CHROM + ")", True)):
    p_ = POS[(POS["chrom"] == TEST_CHROM) == t]
    n_ = match(p_, NEG[(NEG["chrom"] == TEST_CHROM) == t])
    suf = "test" if t else "train"
    write_fa(p_, f"{LOC}.{suf}.pos.fa"); write_fa(n_, f"{LOC}.{suf}.neg.fa")
    rows.append({"split": split, "positives": len(p_), "negatives": len(n_),
                 "GC positives": p_["gc"].mean(), "GC negatives": n_["gc"].mean()})
print(pd.DataFrame(rows).set_index("split").to_string(float_format=lambda x: f"{x:.3f}"))

## 1.3 Train (skipped if the model is already on Drive), and the held-out chromosome-2 AUC

In [ ]:
MODEL = f"{TAG}.model.txt"
if not os.path.exists(MODEL):
    t0 = time.time()
    get_ipython().system(f"{GKM}/gkmtrain {GKM_OPTS} -v 2 {LOC}.train.pos.fa {LOC}.train.neg.fa {LOC}")
    shutil.copy(f"{LOC}.model.txt", MODEL)
    print(f"trained in {(time.time() - t0) / 60:.0f} min")


def gkm_predict(fa, out):
    get_ipython().system(f'{GKM}/gkmpredict -T 4 -v 0 "{fa}" "{MODEL}" "{out}"')
    return pd.read_csv(out, sep="\t", header=None, names=["id", "score"])


sp = gkm_predict(f"{LOC}.test.pos.fa", f"{LOC}.test.pos.out")["score"].values
sn = gkm_predict(f"{LOC}.test.neg.fa", f"{LOC}.test.neg.out")["score"].values
print(f"{CELL} gkm-SVM, held-out {TEST_CHROM}: AUROC {roc_auc_score(np.r_[np.ones(len(sp)), np.zeros(len(sn))], np.r_[sp, sn]):.3f} "
      f"({len(sp):,} peaks vs {len(sn):,} GC-matched background)")

## 1.4 Weights of every 10-mer, and deltaSVM of every Malinois variant

All 524,800 distinct 10-mers (a 10-mer and its reverse complement are one feature) are scored with `gkmpredict`;
deltaSVM = Σ over the 10 windows of length 10 that contain the variant of w(alt) − w(ref), on the hg38 bases
(identical to the MPRA sequence, checked in the first Malinois notebook).

In [ ]:
W_PATH = f"{TAG}.kmer_w.npy"
if not os.path.exists(W_PATH):
    t0 = time.time()
    i = np.arange(4 ** L)
    dig = (i[:, None] >> (2 * np.arange(L)[::-1])) & 3                   # base-4 digits, first base most significant
    rc = ((3 - dig)[:, ::-1] * (4 ** np.arange(L)[::-1])).sum(1)
    keep = np.flatnonzero(i <= rc)
    seqs = ["".join(r) for r in np.array(list("ACGT"))[dig[keep]]]
    with open(f"{LOC}.kmers.fa", "w") as f:
        f.write("".join(f">{j}\n{s}\n" for j, s in zip(keep, seqs)))
    r = gkm_predict(f"{LOC}.kmers.fa", f"{LOC}.kmers.out")
    assert len(r) == len(keep)
    W = np.zeros(4 ** L)
    W[r["id"].values] = r["score"].values
    W[rc[r["id"].values]] = r["score"].values
    np.save(W_PATH, W)
    print(f"{len(keep):,} 10-mers scored in {(time.time() - t0) / 60:.1f} min")
W = np.load(W_PATH)

V = pd.read_csv(f"{MAL}/variants.csv.gz", index_col="vi", dtype={"chrom": str}, low_memory=False,
                usecols=["vi", "chrom38", "pos38", "ref", "alt", f"d_{CELL}", f"se_{CELL}", f"act_{CELL}"])
ENC = np.full(256, -1, np.int64)
for k, b in enumerate("ACGT"):
    ENC[ord(b)] = k
S = np.full((len(V), 2 * L - 1), -1, np.int64)
row = pd.Series(np.arange(len(V)), index=V.index)
for ch, g in V.groupby("chrom38"):
    s = GENOME[ch][:].seq
    for r_, p in zip(row[g.index].values, g["pos38"].values):
        x = s[p - L + 1:p + L]
        if p - L + 1 >= 0 and len(x) == 2 * L - 1:
            S[r_] = ENC[np.frombuffer(x.encode(), np.uint8)]
    del s
REFB = ENC[np.frombuffer("".join(V["ref"]).encode(), np.uint8)]
assert (S[:, L - 1] == REFB).mean() > 0.999, "reference base mismatch"
A = S.copy()
A[:, L - 1] = ENC[np.frombuffer("".join(V["alt"]).encode(), np.uint8)]
PW = 4 ** np.arange(L)[::-1]


def kmer_ids(X):
    win = np.stack([X[:, j:j + L] for j in range(L)], 1)                # (variants, 10 windows, 10 bases)
    bad = (win < 0).any(2).any(1)
    return np.where(win < 0, 0, win) @ PW, bad


ir, bad_r = kmer_ids(S)
ia, bad_a = kmer_ids(A)
DSV = (W[ia] - W[ir]).sum(1)
DSV[bad_r | bad_a | (S[:, L - 1] != REFB)] = np.nan
pd.DataFrame({"vi": V.index, "dsvm": DSV}).to_csv(f"{TAG}.variants.csv", index=False)
ok = np.isfinite(DSV) & np.isfinite(V[f"d_{CELL}"].values)
y = V[f"d_{CELL}"].values
act = V[f"act_{CELL}"].values >= 1
print(f"deltaSVM for {int(np.isfinite(DSV).sum()):,} of {len(V):,} variants, saved to {TAG}.variants.csv")
print(f"quick look, {CELL}: Pearson(deltaSVM, Δ) all {np.corrcoef(DSV[ok], y[ok])[0, 1]:+.3f}, "
      f"active {np.corrcoef(DSV[ok & act], y[ok & act])[0, 1]:+.3f}")

---
# Section 2 — evaluation *(CPU runtime, after Section 1 has finished for all three cells)*

## 2.1 Load

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, time
import numpy as np, pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

MAL = f"{WORK_DIR}/malinois"
POOL = f"{WORK_DIR}/fcav_pool"
DS = f"{MAL}/deltasvm"
OUT = f"{MAL}/eval_deltasvm"
os.makedirs(OUT, exist_ok=True)
CELLS3 = ["HepG2", "K562", "SKNSH"]
TIERS = [("all TFs", 0.0), ("AUC >= 0.70", 0.70), ("AUC >= 0.75", 0.75), ("AUC >= 0.80", 0.80)]
ACTIVE_MIN, Z_CLEAR = 1.0, 2.0
N_PERM, N_BOOT, LAYER = 200, 300, 24
pd.set_option("display.width", 250)

V = pd.read_csv(f"{MAL}/variants.csv.gz", index_col="vi", dtype={"chrom": str}, low_memory=False)
CELLS3 = [c for c in CELLS3 if os.path.exists(f"{DS}/{c}.variants.csv")]       # the cells trained so far
assert CELLS3, "run Section 1 for at least one cell first"
print("cells with deltaSVM:", CELLS3, "" if "SKNSH" in CELLS3 else "(no SK-N-SH yet: the cell contrast is skipped)")
for c in CELLS3:
    V[f"ds_{c}"] = pd.read_csv(f"{DS}/{c}.variants.csv", index_col="vi")["dsvm"].reindex(V.index)
H = pd.read_csv(f"{MAL}/hits.csv")
CALLS = pd.read_csv(f"{MAL}/calls.csv")
PD_ = f"{MAL}/p_nt500m"
PU = pd.concat([pd.read_csv(f"{PD_}/{f}") for f in os.listdir(PD_) if f.endswith(".csv") and not f.endswith(".part.csv")])
H = H[~H["leak"]].merge(PU, on="uid")
PVM = H.groupby(["vi", "motif"])["P"].max().rename("P").reset_index()
LALL = CALLS.merge(PVM, on=["vi", "motif"])
AUC = {}
for m in LALL["motif"].unique():
    z = np.load(f"{POOL}/fits/nt500m/{m}.npz", allow_pickle=False)
    AUC[m] = float(np.asarray(z["auc"])[[int(l) for l in z["layers"]].index(LAYER)])
LALL["auc"] = LALL["motif"].map(AUC)
print(f"{len(V):,} variants; deltaSVM available for", {c: int(V[f'ds_{c}'].notna().sum()) for c in CELLS3})
print(f"{LALL['vi'].nunique():,} variants with a motifbreakR call on a TF with P(bound)")

## 2.2 Functions (as in `Malinois MPRA 2`, with deltaSVM added)

In [ ]:
def rank(x):
    return stats.rankdata(x)


def pear(a, b):
    a, b = a - a.mean(), b - b.mean()
    return float((a * b).sum() / np.sqrt((a * a).sum() * (b * b).sum()))


def zs(y):
    return (y - y.mean()) / y.std()


def recode(g):
    return np.unique(g, return_inverse=True)[1]


def within_tf(v, x, g):
    n = np.bincount(g).astype(float)
    mx, mv = np.bincount(g, x) / n, np.bincount(g, v) / n
    dx, dv = x - mx[g], v - mv[g]
    sxx = np.bincount(g, dx * dx)
    slope = np.divide(np.bincount(g, dx * dv), sxx, out=np.zeros_like(sxx), where=sxx > 1e-12)
    return dv - slope[g] * dx


def fit_within(y, x, p, g):
    # b(x × P) with every TF's own intercept and slope on x
    g = recode(g)
    ry = within_tf(y, x, g)
    X = np.column_stack([within_tf(p, x, g), within_tf(x * p, x, g)])
    b = np.linalg.lstsq(X, ry, rcond=None)[0]
    r = ry - X @ b
    se = np.sqrt(np.diag(np.linalg.inv(X.T @ X)) * (r @ r) / (len(y) - 2 * (g.max() + 1) - 2))
    return b[1], se[1]


def tf_pct(p, g):
    o = np.lexsort((p, g))
    gs = g[o]
    st = np.r_[0, np.flatnonzero(gs[1:] != gs[:-1]) + 1]
    cnt = np.diff(np.r_[st, len(gs)])
    q = np.empty(len(p)); q[o] = (np.arange(len(gs)) - np.repeat(st, cnt) + 0.5) / np.repeat(cnt, cnt)
    return q


def pick(key, cv):
    o = np.lexsort((key, cv))
    return o[np.r_[cv[o][1:] != cv[o][:-1], True]]


def context(L):
    L = L.sort_values(["vi", "motif"]).reset_index(drop=True)
    VI = np.unique(L["vi"].values)
    cv = np.searchsorted(VI, L["vi"].values)
    cm = pd.factorize(L["motif"])[0]
    X = dict(VI=VI, cv=cv, cm=cm, AD=L["alleleDiff"].values, P0=L["P"].values, base=np.argsort(cm, kind="stable"))
    X["FB"] = pick(np.abs(X["AD"]), cv)
    chrom = V.loc[VI, "chrom"].astype(str).values
    X["FOLD"] = np.array([int(ch) % 2 if ch.isdigit() else 0 for ch in chrom])
    X["Y"] = {c: V.loc[VI, f"d_{c}"].values for c in CELLS3}
    X["SE"] = {c: V.loc[VI, f"se_{c}"].values for c in CELLS3}
    X["DS"] = {c: V.loc[VI, f"ds_{c}"].values for c in CELLS3}
    X["SUM_AD"] = np.bincount(cv, X["AD"])
    X["SETS"] = {(c, s): np.isfinite(X["Y"][c]) & np.isfinite(X["DS"][c])
                 & ((V.loc[VI, f"act_{c}"].values >= ACTIVE_MIN) if s == "active" else True)
                 for c in CELLS3 for s in ("all", "active")}
    return X


def shuffle_tf(X, rng):
    o = np.lexsort((rng.random(len(X["P0"])), X["cm"]))
    out = np.empty_like(X["P0"]); out[X["base"]] = X["P0"][o]
    return out


def cv_pear(cols, y, f):
    # linear fit on one chromosome half, predict the other; Pearson over both halves' predictions
    Z = np.column_stack([np.ones(len(y))] + cols)
    out = np.zeros(len(y))
    for k in (0, 1):
        tr, te = f != k, f == k
        out[te] = Z[te] @ np.linalg.lstsq(Z[tr], y[tr], rcond=None)[0]
    return pear(out, y)


def gate_stats(X, p):
    # per (cell, set): [within-TF b(deltaSVM × P), gain of P over deltaSVM, gain of P over deltaSVM + motifbreakR,
    #                   CV Pearson deltaSVM, CV Pearson deltaSVM + motifbreakR]
    FB = X["FB"]
    out = {}
    for (c, s), m in X["SETS"].items():
        y, d, ad = zs(X["Y"][c][m]), X["DS"][c][m], X["AD"][FB][m]
        pf, g, f = p[FB][m], recode(X["cm"][FB][m]), X["FOLD"][m]
        q = tf_pct(pf, g) - 0.5
        bw = fit_within(y, d, pf, g)[0]
        r0, r1 = cv_pear([d], y, f), cv_pear([d, q, d * q], y, f)
        ra, r2 = cv_pear([d, ad], y, f), cv_pear([d, ad, q, ad * q], y, f)
        out[c, s] = np.array([bw, r1 - r0, r2 - ra, r0, ra])
    return out


def detection(X, c, s, rng):
    m = X["SETS"][c, s]
    FB = X["FB"]
    d, p, g, f = np.abs(X["DS"][c][m]), X["P0"][FB][m], recode(X["cm"][FB][m]), X["FOLD"][m]
    clear = (np.abs(X["Y"][c][m]) / X["SE"][c][m] >= Z_CLEAR).astype(int)
    q = tf_pct(p, g)
    p0, p1 = np.zeros(len(d)), np.zeros(len(d))
    for k in (0, 1):
        tr, te = f != k, f == k
        A0, A1 = d[:, None], np.column_stack([d, q, d * q])
        p0[te] = LogisticRegression(max_iter=1000).fit(A0[tr], clear[tr]).decision_function(A0[te])
        p1[te] = LogisticRegression(max_iter=1000).fit(A1[tr], clear[tr]).decision_function(A1[te])
    bs = []
    for _ in range(N_BOOT):
        j = rng.integers(0, len(clear), len(clear))
        bs.append(roc_auc_score(clear[j], p1[j]) - roc_auc_score(clear[j], p0[j]))
    a0 = roc_auc_score(clear, p0)
    return a0, roc_auc_score(clear, p1) - a0, np.percentile(bs, 2.5), np.percentile(bs, 97.5)


def contrast(X, c, s, rng):
    FB = X["FB"]
    PF, TF = X["P0"][FB], X["cm"][FB]
    def one(j, cc):
        mm = j[X["SETS"][cc, s][j]]
        return fit_within(zs(X["Y"][cc][mm]), X["DS"][cc][mm], PF[mm], TF[mm])[0]
    allv = np.arange(len(X["VI"]))
    u = np.flatnonzero(X["SETS"][c, s] | X["SETS"]["SKNSH", s])
    d = [one(j, c) - one(j, "SKNSH") for j in (rng.choice(u, len(u)) for _ in range(N_BOOT))]
    return one(allv, c) - one(allv, "SKNSH"), np.percentile(d, 2.5), np.percentile(d, 97.5)

## 2.3 How good is the baseline? deltaSVM vs motifbreakR (no MPRA training in any of them)

First table: deltaSVM on every variant it can score. Second: each cell's deltaSVM against each cell's Δ (Pearson,
all variants): a cell-specific model should do best on its own cell. Third: on the variants with a motifbreakR call
(the set of 2.4), deltaSVM vs motifbreakR's call with the largest |alleleDiff| and vs alleleDiff summed over calls.

In [ ]:
fmt = lambda x: f"{x:+.3f}"
clear_ = lambda c, m: (V[f"d_{c}"].abs() / V[f"se_{c}"] >= Z_CLEAR).values[m]
rows = []
for c in CELLS3:
    for s in ("all", "active"):
        m = (V[f"d_{c}"].notna() & V[f"ds_{c}"].notna() & ((V[f"act_{c}"] >= ACTIVE_MIN) if s == "active" else True)).values
        y, d = V[f"d_{c}"].values[m], V[f"ds_{c}"].values[m]
        rows.append({"cell": c, "variants": s, "n": int(m.sum()), "Pearson": pear(d, y),
                     "Spearman": pear(rank(d), rank(y)), "AUROC |deltaSVM|, clear effects": roc_auc_score(clear_(c, m), np.abs(d))})
TA = pd.DataFrame(rows).set_index(["cell", "variants"])
print("=== deltaSVM, every variant it scores\n")
print(TA.to_string(float_format=fmt))

M = pd.DataFrame({f"deltaSVM {a}": {f"Δ {c}": pear(*V[[f"ds_{a}", f"d_{c}"]].dropna().values.T) for c in CELLS3} for a in CELLS3})
print("\n=== Pearson of each cell's deltaSVM (columns) with each cell's Δ (rows), all variants\n")
print(M.to_string(float_format=fmt))

X0 = context(LALL)
rows = []
for (c, s), m in X0["SETS"].items():
    y = X0["Y"][c][m]
    clear = np.abs(y) / X0["SE"][c][m] >= Z_CLEAR
    ad, sad, d = X0["AD"][X0["FB"]][m], X0["SUM_AD"][m], X0["DS"][c][m]
    rows.append({"cell": c, "variants": s, "n": int(m.sum()),
                 "Pearson motifbreakR": pear(ad, y), "Pearson summed motifbreakR": pear(sad, y), "Pearson deltaSVM": pear(d, y),
                 "AUROC motifbreakR": roc_auc_score(clear, np.abs(ad)), "AUROC summed": roc_auc_score(clear, np.abs(sad)),
                 "AUROC deltaSVM": roc_auc_score(clear, np.abs(d))})
TB = pd.DataFrame(rows).set_index(["cell", "variants"])
print("\n=== on the variants with a motifbreakR call (all TFs)\n")
print(TB.to_string(float_format=fmt))
TA.to_csv(f"{OUT}/deltasvm_all_variants.csv"); M.to_csv(f"{OUT}/deltasvm_cross_cell.csv"); TB.to_csv(f"{OUT}/baselines_called_variants.csv")

## 2.4 Does P(bound) add to deltaSVM? Every TF tier

* **gain over deltaSVM**: out-of-sample Pearson of Δ ~ deltaSVM + P + deltaSVM × P minus Δ ~ deltaSVM (P as
  within-TF percentile, centred; fitted on one chromosome half, scored on the other).
* **gain over deltaSVM + motifbreakR**: Δ ~ deltaSVM + alleleDiff + P + alleleDiff × P minus Δ ~ deltaSVM + alleleDiff.
* **within-TF b(deltaSVM × P)**: does the CAV amplify deltaSVM's predicted effect, inside a TF.
* p N2 = share of 200 TF-wise shuffles of P at least as large; detection gain and contrast with bootstrap 95% CIs.
About 10–20 minutes.

In [ ]:
NAMES = ["within-TF b(deltaSVM x P)", "gain over deltaSVM", "gain over deltaSVM + motifbreakR"]
ROWS, CROWS = [], []
t_all = time.time()
for name, thr in TIERS:
    X = context(LALL[LALL["auc"] >= thr])
    rng = np.random.default_rng(7)
    obs = gate_stats(X, X["P0"])
    null = [gate_stats(X, shuffle_tf(X, rng)) for _ in range(N_PERM)]
    for key, o in obs.items():
        nl = np.array([n_[key] for n_ in null])
        c, s = key
        a0, gain, lo, hi = detection(X, c, s, rng)
        r = {"threshold": name, "cell": c, "variants": s, "n": int(X["SETS"][key].sum()),
             "CV Pearson deltaSVM": o[3], "CV Pearson deltaSVM + motifbreakR": o[4]}
        for k, nm in enumerate(NAMES):
            r[nm] = o[k]
            r[f"p N2 {k + 1}"] = np.mean(nl[:, k] >= o[k])
        r.update({"AUROC |deltaSVM|": a0, "AUROC gain + P": gain, "AUROC gain CI low": lo, "AUROC gain CI high": hi})
        ROWS.append(r)
    for c in [c for c in ("HepG2", "K562") if c in CELLS3 and "SKNSH" in CELLS3]:
        for s in ("all", "active"):
            o, lo, hi = contrast(X, c, s, rng)
            CROWS.append({"threshold": name, "contrast": f"{c} - SK-N-SH", "variants": s,
                          "difference in within-TF b": o, "95% CI low": lo, "95% CI high": hi})
    print(f"{name}: {len(np.unique(X['cm']))} TFs, {len(X['VI']):,} variants  [{(time.time() - t_all) / 60:.1f} min]", flush=True)
R = pd.DataFrame(ROWS).set_index(["threshold", "cell", "variants"])
CR = pd.DataFrame(CROWS, columns=["threshold", "contrast", "variants", "difference in within-TF b", "95% CI low",
                                  "95% CI high"]).set_index(["threshold", "contrast", "variants"])
R.to_csv(f"{OUT}/p_on_top_of_deltasvm.csv"); CR.to_csv(f"{OUT}/p_on_top_contrast.csv")
print(f"\np N2 k = share of {N_PERM} TF-wise shuffles of P at least as large as statistic k\n")
print("=== prediction: P on top of deltaSVM (out-of-sample Pearson)")
print(R[["n", "CV Pearson deltaSVM", NAMES[1], "p N2 2", "CV Pearson deltaSVM + motifbreakR", NAMES[2], "p N2 3"]].to_string(float_format=fmt))
print("\n=== within-TF amplification and detection of clear effects")
print(R[[NAMES[0], "p N2 1", "AUROC |deltaSVM|", "AUROC gain + P", "AUROC gain CI low", "AUROC gain CI high"]].to_string(float_format=fmt))
print("\n=== cell contrast of the within-TF b(deltaSVM x P) (bootstrap 95% CI)")
print(CR.to_string(float_format=fmt))

## End

Outputs in `malinois/eval_deltasvm/`: `deltasvm_all_variants.csv`, `deltasvm_cross_cell.csv`,
`baselines_called_variants.csv`, `p_on_top_of_deltasvm.csv`, `p_on_top_contrast.csv`. The deltaSVM models, 10-mer
weights and variant scores are in `malinois/deltasvm/`.